In [5]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from moviepy.video.io.VideoFileClip import VideoFileClip
import os
import glob

In [6]:
model_path = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"

processor = AutoProcessor.from_pretrained(model_path)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.float32,
    _attn_implementation="eager"
).to("cpu")

model.eval()


SmolVLMForConditionalGeneration(
  (model): SmolVLMModel(
    (vision_model): SmolVLMVisionTransformer(
      (embeddings): SmolVLMVisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
        (position_embedding): Embedding(1024, 768)
      )
      (encoder): SmolVLMEncoder(
        (layers): ModuleList(
          (0-11): 12 x SmolVLMEncoderLayer(
            (self_attn): SmolVLMVisionAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): SmolVLMVisionMLP(
              (activation_fn): PytorchGELUTanh()
              (fc1): Linear(in_features=768, out_

In [7]:
SYSTEM = (
    "You are an event detection model. "
    "Your task is to identify only the salient visible events in the video. "
    "Do not describe background details unless they are necessary for understanding an event. "
    "Do not repeat the same event. "
    "Return the answer as a numbered list."
)

#1) Retrieve all salient events in the video. Shit for both video 5 and video 6!
#2) Describe all the key events in the video. Much better for video 5 and incredibly better for video 6. A lot of yapping though and overlap.
#3) "Describe all the key events in the video." "Return only events that are important for understanding the video." A bit better, but still hasn't captured salient events.

PROMPT_EVENT_ONLY = ("""
You are analyzing a low-quality video made of sampled frames.

Your task is to identify key events events: important visible actions or changes over time.
Focus on what changes between frames, not on static background details.   
""")

PROMPT_EVENT_TIMESTAMP = (
"""
Describe all the key events in the video with approximate timestamps.
"""
)

def run_vlm_on_video(video_path, prompt, max_frames=16, max_new_tokens=500):
    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "path": video_path},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        video_load_backend="pyav",
        max_frames=max_frames,
    )

    inputs = {
        k: v.to("cpu") if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens
        )

    generated_only = generated_ids[:, inputs["input_ids"].shape[1]:]

    answer = processor.batch_decode(
        generated_only,
        skip_special_tokens=True
    )[0]

    return answer.strip()

In [8]:
def split_video_into_chunks(video_path, output_dir, chunk_length=30):
    os.makedirs(output_dir, exist_ok=True)

    # If chunks already exist, reuse them
    existing_chunks = sorted(glob.glob(os.path.join(output_dir, "chunk_*.mp4")))
    if existing_chunks:
        chunk_paths = []

        for chunk_path in existing_chunks:
            filename = os.path.basename(chunk_path)
            parts = filename.replace("chunk_", "").replace(".mp4", "").split("_")
            start = int(parts[0])
            end = int(parts[1])
            chunk_paths.append((chunk_path, start, end))

        print(f"Reusing {len(chunk_paths)} existing chunks from {output_dir}")
        return chunk_paths

    # Otherwise, create chunks
    video = VideoFileClip(video_path)
    duration = int(video.duration)

    chunk_paths = []

    for start in range(0, duration, chunk_length):
        end = min(start + chunk_length, duration)

        chunk_path = os.path.join(
            output_dir,
            f"chunk_{start:04d}_{end:04d}.mp4"
        )

        if hasattr(video, "subclipped"):
            clip = video.subclipped(start, end)
        else:
            clip = video.subclip(start, end)

        clip.write_videofile(
            chunk_path,
            codec="libx264",
            audio=False,
            logger=None
        )

        clip.close()
        chunk_paths.append((chunk_path, start, end))

    video.close()
    return chunk_paths

In [9]:
video_path = "TVSum/video_6.mp4"

chunks = split_video_into_chunks(
    video_path,
    output_dir="video_6_chunks",
    chunk_length=30
)

all_outputs = []

for chunk_path, start, end in chunks:
    output = run_vlm_on_video(
        chunk_path,
        PROMPT_EVENT_ONLY,
        max_frames=16,
        max_new_tokens=500
    )

    all_outputs.append({
        "chunk": chunk_path,
        "start": start,
        "end": end,
        "vlm_output": output
    })

    print(f"\nCHUNK {start}-{end} seconds")
    print(output)

Reusing 11 existing chunks from video_6_chunks

CHUNK 0-30 seconds
The video shows a person walking down a sidewalk, then a car drives by.

CHUNK 30-60 seconds
The truck is moving down the road.

CHUNK 60-90 seconds
The truck is moving down the road, and the camera captures the truck's movement and the surrounding environment.

CHUNK 90-120 seconds
The truck is moving slowly down the road.

CHUNK 120-150 seconds
The video shows a blue and white bus driving down a dirt road in a forest. The bus is moving forward, and the road is surrounded by dense foliage. The bus is moving from left to right, and the camera angle is slightly tilted, giving a sense of the bus's speed and direction. The bus is moving from the left to the right, and the road is surrounded by dense foliage. The bus is moving from the left to the right, and the road is surrounded by dense foliage. The bus is moving from the left to the right, and the road is surrounded by dense foliage. The bus is moving from the left to t

In [ ]:
import re

def parse_vlm_events(text):
    """
    Extracts events from VLM output formatted like:
    Salient event 1: A person enters the room
    Salient event 2: A person sits down, 00:03-00:07
    """

    pattern = r"Salient event\s*\d+\s*:\s*(.*?)(?=\n\s*Salient event\s*\d+\s*:|\Z)"
    matches = re.findall(pattern, text, flags=re.IGNORECASE | re.DOTALL)

    parsed_events = []

    for match in matches:
        event_text = match.strip().replace("\n", " ")

        time_pattern = r"(\d{1,2}:\d{2})\s*[-–]\s*(\d{1,2}:\d{2})"
        time_match = re.search(time_pattern, event_text)

        if time_match:
            start_time = time_match.group(1)
            end_time = time_match.group(2)
            event_description = re.sub(time_pattern, "", event_text).strip(" ,.-")
        else:
            start_time = None
            end_time = None
            event_description = event_text.strip(" ,.-")

        parsed_events.append({
            "event": event_description,
            "start_time": start_time,
            "end_time": end_time
        })

    return parsed_events

In [12]:
parsed_event_only = parse_vlm_events(event_only_output)
parsed_event_timestamp = parse_vlm_events(event_timestamp_output)

print(parsed_event_only)
print(parsed_event_timestamp)

[]
[]
